<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/project-2-llm-evaluation-suite/05c_mitre_atlas_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 5c: MITRE ATLAS v5.4.0 Technique Mapping and NIST AI RMF Compliance Report Card

**Goal:** Map Phase 5a (OWASP LLM Top 10) and Phase 5b (OWASP Agentic Top 10) attack results onto MITRE ATLAS v5.4.0 techniques using a deterministic lookup table, not a model judgment call, since the OWASP-category-to-ATLAS-technique relationship is a fixed taxonomy, not something requiring case-by-case interpretation. Generate a NIST AI RMF compliance report card summarizing detection coverage.

**Design decision, stated plainly:** unlike 05a/05b's attack detection logic, this notebook's core mapping requires no API call at all, live or simulated. A lookup table is not a placeholder for a future real implementation, it is the correct, permanent implementation. This is why this notebook has no `SIMULATED_OUTPUT` flag for the mapping step itself.

**Honest sourcing caveat:** there is no single official crosswalk published jointly by OWASP and MITRE. The mappings below are synthesized from MITRE's own published ATLAS technique descriptions, cross-checked against multiple independent security-industry writeups, not copied from one authoritative source. Each mapping below is labeled by confidence: **direct** (name and scope closely match, corroborated across multiple sources), **interpretive** (a reasonable synthesis, not universally published this way), or **unresolved** (no clean existing technique identified). This labeling is itself part of the deliverable, not a hedge to remove later.

**Tools:** MITRE ATLAS v5.4.0 technique taxonomy (AML.T-prefixed technique IDs), NIST AI RMF

**Date:** July 2026

In [1]:
# Cell 2: Mount Drive and confirm Phase 5a and 5b

from google.colab import drive
drive.mount('/content/drive')

import os, json

DRIVE_PATH = "/content/drive/MyDrive/python-ai-governance-p2/data/"

phase5a_path = DRIVE_PATH + "phase05a_promptfoo_owasp_results.json"
phase5b_path = DRIVE_PATH + "phase05b_promptfoo_owasp_agentic_results.json"

missing = []
if os.path.exists(phase5a_path):
    with open(phase5a_path) as f:
        phase5a = json.load(f)
    print("Phase 5a results confirmed.")
    print(f"  Detection rate: {phase5a['detection_rate']:.0%} "
          f"({phase5a['detected_count']}/{phase5a['attack_case_count']})")
else:
    missing.append(phase5a_path)

if os.path.exists(phase5b_path):
    with open(phase5b_path) as f:
        phase5b = json.load(f)
    print("Phase 5b results confirmed.")
    print(f"  Detection rate: {phase5b['detection_rate']:.0%} "
          f"({phase5b['detected_count']}/{phase5b['attack_case_count']})")
else:
    missing.append(phase5b_path)

if missing:
    print("WARNING: missing files:")
    for m in missing:
        print(" ", m)
    print("Run 05a_promptfoo_owasp_llm.ipynb and/or 05b_promptfoo_owasp_agentic.ipynb first.")

Mounted at /content/drive
Phase 5a results confirmed.
  Detection rate: 100% (10/10)
Phase 5b results confirmed.
  Detection rate: 100% (12/12)


In [2]:
# Cell 3: Install packages
# This notebook's core work, the ATLAS mapping and the NIST report card, is
# deterministic and needs no LLM API at all. The only install here is
# Langfuse, kept for consistency with every other phase's trace logging,
# and pandas, used only to format the report card table.

!pip install langfuse pandas --quiet

print("Packages installed.")
print("No Gemini or Claude client is needed in this notebook's core logic.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
Packages installed.
No Gemini or Claude client is needed in this notebook's core logic.


In [3]:
# Cell 4: Simulated output flag and Langfuse setup
# SIMULATED_OUTPUT here only governs whether Langfuse traces are sent live
# or recorded locally, for consistency with every other phase's logging
# convention. It has no effect on the ATLAS mapping or the report card,
# both of which are deterministic and run identically either way.

SIMULATED_OUTPUT = True

from google.colab import userdata

if not SIMULATED_OUTPUT:
    from langfuse import Langfuse
    langfuse = Langfuse(
        public_key=userdata.get('LANGFUSE_PUBLIC_KEY'),
        secret_key=userdata.get('LANGFUSE_SECRET_KEY'),
        host="https://cloud.langfuse.com"
    )
    print("Langfuse client initialised.")
else:
    print("[SIMULATED] Langfuse client not initialised (trace logging only).")
    print(f"SIMULATED_OUTPUT = {SIMULATED_OUTPUT}")

print()
print("Note: the ATLAS mapping and NIST report card below are deterministic")
print("lookup tables, not judgment calls. They run identically regardless")
print("of SIMULATED_OUTPUT.")

[SIMULATED] Langfuse client not initialised (trace logging only).
SIMULATED_OUTPUT = True

Note: the ATLAS mapping and NIST report card below are deterministic
lookup tables, not judgment calls. They run identically regardless
of SIMULATED_OUTPUT.


In [4]:
# Cell 5: MITRE ATLAS v5.4.0 technique lookup table
# Deterministic mapping, not a model judgment call. Built from MITRE's own
# published ATLAS technique descriptions, cross-checked against independent
# security-industry sources, not copied from one single authority, since no
# official joint OWASP/MITRE crosswalk exists.
#
# confidence:
#   "direct"       - name and scope closely match, corroborated across
#                    multiple independent sources
#   "interpretive" - a reasonable synthesis, not universally published
#                    this exact way
#   "unresolved"   - no clean existing ATLAS technique identified

ATLAS_MAPPING = {
    # --- OWASP LLM Top 10 (2025), Phase 5a ---
    "owasp_llm01": {
        "technique_id": "AML.T0051",
        "technique_name": "LLM Prompt Injection",
        "confidence": "direct",
        "note": "Directly named and widely corroborated as the ATLAS technique for OWASP LLM01."
    },
    "owasp_llm02": {
        "technique_id": "AML.T0024",
        "technique_name": "Exfiltration via ML Inference API",
        "confidence": "interpretive",
        "note": "Insecure output handling often results in downstream systems trusting "
                "unsanitised LLM output, which overlaps with inference-API exfiltration "
                "patterns, but ATLAS has no technique named for output handling specifically."
    },
    "owasp_llm03": {
        "technique_id": "AML.T0020",
        "technique_name": "Poison Training Data",
        "confidence": "direct",
        "note": "Name and scope match directly."
    },
    "owasp_llm04": {
        "technique_id": "AML.T0029",
        "technique_name": "Denial of ML Service",
        "secondary_technique_id": "AML.T0034",
        "secondary_technique_name": "Cost Harvesting",
        "confidence": "direct",
        "note": "Two ATLAS techniques both apply: service disruption and cost-based abuse."
    },
    "owasp_llm05": {
        "technique_id": "AML.T0010",
        "technique_name": "ML Supply Chain Compromise",
        "confidence": "direct",
        "note": "Name and scope match directly."
    },
    "owasp_llm06": {
        "technique_id": "AML.T0024",
        "technique_name": "Exfiltration via ML Inference API",
        "confidence": "direct",
        "note": "Direct match for sensitive information disclosure via inference."
    },
    "owasp_llm07": {
        "technique_id": "AML.T0053",
        "technique_name": "LLM Plugin Compromise",
        "confidence": "direct",
        "note": "Name and scope match directly."
    },
    "owasp_llm08": {
        "technique_id": "AML.T0053",
        "technique_name": "LLM Plugin Compromise",
        "confidence": "interpretive",
        "note": "Excessive agency and insecure plugin design overlap significantly in ATLAS's "
                "current taxonomy; no distinct technique isolates unauthorised agent action "
                "specifically."
    },
    "owasp_llm09": {
        "technique_id": "AML.T0062",
        "technique_name": "Discover LLM Hallucinations",
        "confidence": "interpretive",
        "note": "ATLAS's technique concerns an attacker discovering and exploiting "
                "hallucination as a staging step, not a system's users overrelying on output. "
                "Related but not a precise match."
    },
    "owasp_llm10": {
        "technique_id": "AML.T0044",
        "technique_name": "Full ML Model Access",
        "confidence": "direct",
        "note": "Name and scope match directly."
    },

    # --- OWASP Top 10 for Agentic Applications (2026), Phase 5b ---
    "owasp_aai01": {
        "technique_id": "AML.TA0015",
        "technique_name": "Command and Control (tactic, added ATLAS v5.1.0, Nov 2025)",
        "confidence": "interpretive",
        "note": "Authorization hijacking maps to the broader Command and Control tactic "
                "added for agent security; ATLAS's agent-specific technique granularity "
                "under this tactic is still maturing as of this writing."
    },
    "owasp_aai02": {
        "technique_id": "AML.T0053",
        "technique_name": "LLM Plugin Compromise",
        "confidence": "interpretive",
        "note": "Critical-systems interaction typically occurs through a compromised "
                "tool/plugin path in current agent architectures."
    },
    "owasp_aai03": {
        "technique_id": "AML.T0051",
        "technique_name": "LLM Prompt Injection",
        "confidence": "direct",
        "note": "Goal and instruction manipulation is prompt injection applied to an "
                "agent's stated goal rather than a single response."
    },
    "owasp_aai04": {
        "technique_id": "AML.T0062",
        "technique_name": "Discover LLM Hallucinations",
        "confidence": "direct",
        "note": "Direct match, this is the named ATLAS technique for this exact behavior."
    },
    "owasp_aai05": {
        "technique_id": "AML.T0048",
        "technique_name": "ML Supply Chain Compromise (Impact family)",
        "confidence": "interpretive",
        "note": "Blast-radius/impact-chain concerns are described across the AML.T0048 "
                "series as consequences rather than a single named technique."
    },
    "owasp_aai06": {
        "technique_id": "AML.T0054",
        "technique_name": "LLM Jailbreak / Indirect Prompt Injection (naming inconsistent across sources)",
        "confidence": "unresolved",
        "note": "Independent sources label AML.T0054 differently (one names it 'LLM "
                "Jailbreak', another 'Indirect Prompt Injection'). Flagged as unresolved "
                "rather than asserting one name with false confidence."
    },
    "owasp_aai07": {
        "technique_id": "AML.TA0015",
        "technique_name": "Command and Control (tactic)",
        "confidence": "interpretive",
        "note": "Same tactic-level mapping as AAI01. No distinct multi-agent-orchestration "
                "technique identified in sources reviewed."
    },
    "owasp_aai08": {
        "technique_id": "AML.T0029",
        "technique_name": "Denial of ML Service",
        "confidence": "direct",
        "note": "Direct match, same technique as owasp_llm04's primary mapping."
    },
    "owasp_aai09": {
        "technique_id": "AML.T0010",
        "technique_name": "ML Supply Chain Compromise",
        "confidence": "direct",
        "note": "Direct match, same technique as owasp_llm05."
    },
    "owasp_aai10": {
        "technique_id": None,
        "technique_name": None,
        "confidence": "unresolved",
        "note": "No existing ATLAS technique specifically addressing agent audit-trail "
                "suppression or untraceability was identified in sources reviewed. Reported "
                "as an honest gap rather than forced into an unrelated technique."
    },
}

direct_count = sum(1 for v in ATLAS_MAPPING.values() if v["confidence"] == "direct")
interpretive_count = sum(1 for v in ATLAS_MAPPING.values() if v["confidence"] == "interpretive")
unresolved_count = sum(1 for v in ATLAS_MAPPING.values() if v["confidence"] == "unresolved")

print(f"ATLAS mapping table loaded: {len(ATLAS_MAPPING)} categories.")
print(f"  Direct matches:       {direct_count}")
print(f"  Interpretive matches: {interpretive_count}")
print(f"  Unresolved:           {unresolved_count}")

ATLAS mapping table loaded: 20 categories.
  Direct matches:       11
  Interpretive matches: 7
  Unresolved:           2


In [5]:
# Cell 6: Apply the ATLAS mapping to Phase 5a and 5b results

def map_result_to_atlas(result: dict) -> dict:
    """Attaches the deterministic ATLAS mapping to a single attack result.
    No judgment involved, this is a dictionary lookup."""
    mapping = ATLAS_MAPPING.get(result["id"])
    if mapping is None:
        return {
            **result,
            "atlas_technique_id": None,
            "atlas_technique_name": None,
            "atlas_confidence": "no_mapping_defined",
        }
    return {
        **result,
        "atlas_technique_id": mapping["technique_id"],
        "atlas_technique_name": mapping["technique_name"],
        "atlas_confidence": mapping["confidence"],
    }


mapped_5a = [map_result_to_atlas(r) for r in phase5a["per_case_results"]]
mapped_5b = [map_result_to_atlas(r) for r in phase5b["per_case_results"]]

all_mapped = mapped_5a + mapped_5b

print("ATLAS-MAPPED RESULTS")
print("=" * 70)
for r in all_mapped:
    conf_flag = {"direct": "", "interpretive": " (interpretive)", "unresolved": " (UNRESOLVED)"}
    conf = r.get("atlas_confidence", "no_mapping_defined")
    flag = conf_flag.get(conf, "")
    print(f"  {r['id']}: {r.get('atlas_technique_id', 'N/A')} "
          f"{r.get('atlas_technique_name', '')}{flag}")

print()
print(f"Total mapped: {len(all_mapped)} (10 from Phase 5a + 12 from Phase 5b)")

ATLAS-MAPPED RESULTS
  owasp_llm01: AML.T0051 LLM Prompt Injection
  owasp_llm02: AML.T0024 Exfiltration via ML Inference API (interpretive)
  owasp_llm03: AML.T0020 Poison Training Data
  owasp_llm04: AML.T0029 Denial of ML Service
  owasp_llm05: AML.T0010 ML Supply Chain Compromise
  owasp_llm06: AML.T0024 Exfiltration via ML Inference API
  owasp_llm07: AML.T0053 LLM Plugin Compromise
  owasp_llm08: AML.T0053 LLM Plugin Compromise (interpretive)
  owasp_llm09: AML.T0062 Discover LLM Hallucinations (interpretive)
  owasp_llm10: AML.T0044 Full ML Model Access
  owasp_aai01: AML.TA0015 Command and Control (tactic, added ATLAS v5.1.0, Nov 2025) (interpretive)
  owasp_aai02: AML.T0053 LLM Plugin Compromise (interpretive)
  owasp_aai03: AML.T0051 LLM Prompt Injection
  owasp_aai04: AML.T0062 Discover LLM Hallucinations
  owasp_aai05: AML.T0048 ML Supply Chain Compromise (Impact family) (interpretive)
  owasp_aai06: AML.T0054 LLM Jailbreak / Indirect Prompt Injection (naming inconsistent a

In [6]:
# Cell 7: Detection breakdown by ATLAS confidence tier
# This checks whether detection performance differs across mapping
# confidence tiers, a useful sanity check: if all detections cluster in
# "direct" mappings and misses cluster in "interpretive" or "unresolved"
# ones, that would suggest the underlying attack categories genuinely are
# harder to defend against, not just harder to label.

from collections import defaultdict

by_confidence = defaultdict(list)
for r in all_mapped:
    by_confidence[r.get("atlas_confidence", "no_mapping_defined")].append(r)

print("DETECTION RATE BY ATLAS MAPPING CONFIDENCE")
print("=" * 70)
for conf_tier in ["direct", "interpretive", "unresolved"]:
    cases = by_confidence.get(conf_tier, [])
    if not cases:
        continue
    detected = sum(1 for r in cases if r["detected"])
    print(f"  {conf_tier}: {detected}/{len(cases)} detected "
          f"({detected/len(cases):.0%})")

print()
overall_detected = sum(1 for r in all_mapped if r["detected"])
print(f"Overall across both phases: {overall_detected}/{len(all_mapped)} "
      f"({overall_detected/len(all_mapped):.0%})")
print()
print("Interpretation: if detection rate is roughly equal across confidence "
      "tiers, that suggests the mapping-table uncertainty is independent of "
      "actual pipeline defense, i.e. we're just as unsure how to label the "
      "technique as we are confident the pipeline handled it. If detection "
      "clusters differently by tier, that's worth investigating rather than "
      "treated as coincidence.")

DETECTION RATE BY ATLAS MAPPING CONFIDENCE
  direct: 11/11 detected (100%)
  interpretive: 7/7 detected (100%)
  unresolved: 2/2 detected (100%)

Overall across both phases: 22/22 (100%)

Interpretation: if detection rate is roughly equal across confidence tiers, that suggests the mapping-table uncertainty is independent of actual pipeline defense, i.e. we're just as unsure how to label the technique as we are confident the pipeline handled it. If detection clusters differently by tier, that's worth investigating rather than treated as coincidence.


In [7]:
# Cell 8: NIST AI RMF compliance report card
# Deterministic synthesis of the detection results into NIST AI RMF's four
# functions (Govern, Map, Measure, Manage). No model call needed here either,
# this is a structured summary of numbers already computed above.
#
# Correction to Cell 7's printed interpretation: a 22/22 clean sweep across
# all confidence tiers provides no real signal about whether mapping
# confidence is independent of detection performance, since there were no
# misses to compare against. That interpretation was written anticipating
# variance that did not occur, and the absence of variance should not be
# read as confirmation. This is also a reminder that all 22 "detections"
# come from the same simulated substring/keyword heuristics flagged
# throughout Phase 5a and 5b, not real judgment calls, so a clean sweep is
# expected of the heuristic, not necessarily informative about the pipeline.

import pandas as pd

report_card = {
    "GOVERN": {
        "description": "Policies and accountability for AI risk management",
        "evidence": "Three-queue routing (Federico design addition, Phase 3) "
                    "assigns clear ownership for borderline vs confident-fail "
                    "cases. Judge-model independence (Claude judging Gemini) "
                    "established as a structural policy since Phase 2.",
        "status": "PARTIALLY EVIDENCED",
        "gap": "No live governance sign-off workflow has been exercised yet, "
               "only designed."
    },
    "MAP": {
        "description": "Understanding context and identifying risks",
        "evidence": f"20 OWASP categories (10 LLM Top 10, 10 Agentic Top 10) "
                     f"mapped to {direct_count} direct, {interpretive_count} "
                     f"interpretive, and {unresolved_count} unresolved MITRE "
                     f"ATLAS techniques.",
        "status": "EVIDENCED",
        "gap": f"{unresolved_count} categories (owasp_aai06, owasp_aai10) have "
               f"no clean ATLAS technique identified, an honest gap in the "
               f"current taxonomy, not a gap in this project's mapping effort."
    },
    "MEASURE": {
        "description": "Testing, evaluation, verification, and validation",
        "evidence": f"22 attack cases evaluated across Phase 5a and 5b, "
                     f"{overall_detected}/{len(all_mapped)} detected in this "
                     f"simulated pass.",
        "status": "SIMULATED ONLY",
        "gap": "All 22 detections rely on substring/keyword heuristics, not "
               "real semantic judgment. No live Gemini or Claude call has "
               "been made in the detection logic itself. This is the single "
               "largest gap in this report card and should not be understated."
    },
    "MANAGE": {
        "description": "Risk response and ongoing monitoring",
        "evidence": "Langfuse trace logging structure in place for every "
                     "attack case (Phase 5a Cell 10, Phase 5b Cell 10). "
                     "Three-queue routing provides a designed escalation path.",
        "status": "PARTIALLY EVIDENCED",
        "gap": "No live trace has actually been sent to Langfuse yet "
               "(SIMULATED_OUTPUT = True throughout). No regression alarm "
               "exists yet, that is Phase 6b's scope."
    }
}

df = pd.DataFrame(report_card).T
df.index.name = "NIST AI RMF Function"
print(df.to_string())

                                                             description                                                                                                                                                                                                                    evidence               status                                                                                                                                                                                                                                               gap
NIST AI RMF Function                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [8]:
# Cell 9: Langfuse trace logging

def create_trace(name: str, metadata: dict) -> dict:
    trace = {"name": name, "metadata": metadata, "scores": []}
    if not SIMULATED_OUTPUT:
        lf_trace = langfuse.trace(name=name, metadata=metadata)
        trace["langfuse_id"] = lf_trace.id
    else:
        trace["langfuse_id"] = f"simulated-{name}"
    return trace


def log_score(trace: dict, name: str,
              value: float, comment: str = "") -> None:
    trace["scores"].append({
        "name": name,
        "value": round(value, 4),
        "comment": comment
    })
    if not SIMULATED_OUTPUT:
        langfuse.score(
            trace_id=trace["langfuse_id"],
            name=name,
            value=value,
            comment=comment
        )


traces_5c = []
for r in all_mapped:
    trace = create_trace(
        name=f"phase05c_{r['id']}",
        metadata={
            "phase": "05c",
            "notebook": "05c_mitre_atlas_mapping",
            "atlas_technique_id": r.get("atlas_technique_id"),
            "atlas_technique_name": r.get("atlas_technique_name"),
            "atlas_confidence": r.get("atlas_confidence"),
            "detected": r["detected"],
            "simulated": SIMULATED_OUTPUT
        }
    )
    log_score(
        trace,
        "phase_05c_atlas_mapping_confidence",
        {"direct": 1.0, "interpretive": 0.5, "unresolved": 0.0}.get(r.get("atlas_confidence"), 0.0),
        f"{r.get('atlas_technique_id', 'N/A')}: {r.get('atlas_technique_name', 'unresolved')}"
    )
    traces_5c.append(trace)

summary_trace_5c = create_trace(
    name="phase05c_suite_summary",
    metadata={
        "phase": "05c",
        "total_mapped": len(all_mapped),
        "direct_count": direct_count,
        "interpretive_count": interpretive_count,
        "unresolved_count": unresolved_count,
        "simulated": SIMULATED_OUTPUT
    }
)
log_score(summary_trace_5c, "phase_05c_mapping_coverage",
          direct_count / len(ATLAS_MAPPING),
          f"{direct_count} of {len(ATLAS_MAPPING)} categories have direct ATLAS mapping")

print(f"Traces logged: {len(traces_5c)} mapping traces + 1 summary")
print(f"Summary trace: {summary_trace_5c['langfuse_id']}")

Traces logged: 22 mapping traces + 1 summary
Summary trace: simulated-phase05c_suite_summary


In [9]:
# Cell 10: Save results to Drive

import json
from datetime import datetime

output_5c = {
    "phase": "05c_mitre_atlas_mapping",
    "timestamp": datetime.now().isoformat(),
    "atlas_version": "v5.4.0",
    "mapping_methodology": (
        "Deterministic lookup table, not a model judgment call. No official "
        "joint OWASP/MITRE crosswalk exists; mappings synthesized from "
        "MITRE's published ATLAS technique descriptions, cross-checked "
        "against independent security-industry sources."
    ),
    "mapping_confidence_breakdown": {
        "direct": direct_count,
        "interpretive": interpretive_count,
        "unresolved": unresolved_count,
        "total_categories": len(ATLAS_MAPPING)
    },
    "atlas_mapping_table": ATLAS_MAPPING,
    "mapped_results": all_mapped,
    "detection_by_confidence_tier": {
        conf: {
            "detected": sum(1 for r in cases if r["detected"]),
            "total": len(cases)
        }
        for conf, cases in by_confidence.items()
    },
    "nist_rmf_report_card": report_card,
    "langfuse_summary_trace": summary_trace_5c["langfuse_id"],
    "design_notes": {
        "cell_7_correction": (
            "Cell 7's printed interpretation anticipated that detection rate "
            "might vary by ATLAS mapping confidence tier. A 22/22 clean "
            "sweep across all tiers occurred instead, which provides no "
            "real signal either way, there was no variance to analyze. This "
            "is stated explicitly rather than letting the original "
            "interpretation stand as if it had been confirmed."
        ),
        "simulation_status": (
            "The ATLAS mapping and NIST report card generation are genuinely "
            "deterministic and require no live API call, this is real, "
            "permanent logic, not a placeholder. The 22/22 detection input "
            "feeding into this mapping, however, still comes from Phase 5a "
            "and 5b's simulated substring/keyword heuristics, not real "
            "judgment. The mapping is real; the detections it is mapping "
            "are not yet."
        )
    }
}

output_path = DRIVE_PATH + "phase05c_mitre_atlas_mapping_results.json"
with open(output_path, "w") as f:
    json.dump(output_5c, f, indent=2)

print(f"Results saved: {output_path}")
print()
print("Summary:")
print(f"  Categories mapped: {len(ATLAS_MAPPING)}")
print(f"  Direct / Interpretive / Unresolved: "
      f"{direct_count} / {interpretive_count} / {unresolved_count}")
print(f"  Attack cases mapped: {len(all_mapped)}")

Results saved: /content/drive/MyDrive/python-ai-governance-p2/data/phase05c_mitre_atlas_mapping_results.json

Summary:
  Categories mapped: 20
  Direct / Interpretive / Unresolved: 11 / 7 / 2
  Attack cases mapped: 22


## Phase 5c Findings: MITRE ATLAS v5.4.0 Technique Mapping and NIST AI RMF Compliance Report Card

**Frameworks:** MITRE ATLAS v5.4.0, NIST AI RMF (Govern, Map, Measure, Manage)

**What was built:** A deterministic lookup table mapping all 20 OWASP categories (10 LLM Top 10 from Phase 5a, 10 Agentic Top 10 from Phase 5b) to MITRE ATLAS technique IDs, with each mapping labeled by confidence rather than presented as uniformly authoritative. Applied to all 22 attack cases from both prior phases. A NIST AI RMF compliance report card synthesizing detection results, mapping coverage, and Langfuse trace status across all four RMF functions.

**What was found:**

| Confidence Tier | Count | Detection Rate |
|---|---|---|
| Direct | 11 | 11/11 (100%) |
| Interpretive | 7 | 7/7 (100%) |
| Unresolved | 2 | 2/2 (100%) |

Of 20 OWASP categories, 11 have a direct, well-corroborated MITRE ATLAS technique match. 7 required interpretive synthesis, since no official OWASP/MITRE crosswalk exists. 2 (`owasp_aai06`, memory/context manipulation, and `owasp_aai10`, agent untraceability) have no clean existing ATLAS technique identified at all, an honest gap in the current taxonomy's coverage of newer agentic attack patterns, not a gap in this project's research effort.

**Correction, stated directly rather than left standing:** Cell 7's printed interpretation speculated about what a difference in detection rate across confidence tiers would mean. No difference occurred, all three tiers hit 100%, which means that interpretation was never actually tested and should not be read as confirmed. A clean sweep with no variance tells us nothing about whether mapping confidence correlates with detection performance, it only tells us there were no misses to compare.

**The finding that matters most in this notebook:** the ATLAS mapping and NIST report card generation are genuinely real, permanent logic, not simulated placeholders, they require no live model call and would produce the identical output today, tomorrow, or after Gemini and Claude billing is funded. This is a deliberate design difference from every prior Phase 5 notebook. What is *not* yet real is the detection data being fed into that mapping: all 22 "detected" outcomes still come from Phase 5a and 5b's substring/keyword heuristics, not actual semantic judgment. The mapping logic is production-ready; the inputs it maps are not, yet.

**NIST AI RMF report card summary:** GOVERN and MANAGE are partially evidenced (designed but not yet exercised live). MAP is evidenced (the mapping itself is complete and real). MEASURE is explicitly marked simulated only, and flagged as the report card's single largest gap, since detection results throughout are heuristic, not judged.

**Talabat connection:** this notebook demonstrates a specific, useful distinction for a governance engineering role: knowing which parts of a compliance pipeline are genuinely deterministic and production-ready today (framework mapping, report card synthesis) versus which parts require live model judgment and real API access (the actual detection layer). Conflating the two, treating a lookup table's completeness as if it proves the detection layer works, would be exactly the kind of overclaiming this project's own house rules and the Talabat prep document both warn against.

**Simulated output note:** `SIMULATED_OUTPUT` in this notebook governs only whether Langfuse traces are sent live; it has no bearing on the ATLAS mapping or NIST report card, both of which are real, deterministic code with no simulated branch at all.

**Next step:** Phase 6a (`06a_langfuse_custom_scores.ipynb`) wires RAGAS, DeepEval, and Promptfoo scores into Langfuse as structured custom scores on traced runs, and is being built, per the correction made during Phase 5b, with a real Claude-judge implementation from the start rather than a simulated placeholder.